# Wake Word Quickstart — Kaggle / Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/kaggle_quickstart.ipynb)

> **Zero to ONNX in one notebook.**  \
> Type a wake phrase, click **Run All**, and this notebook produces two ONNX files you can\
> drop straight into an [OpenVoiceOS](https://openvoiceos.org) (or Rhasspy) device.\
> No local GPU required — the free Kaggle T4 tier is enough.

---

## Prerequisites

### Running on Kaggle (recommended — free GPU)

1. Create a free [Kaggle account](https://www.kaggle.com) if you do not have one.
2. Open this notebook in Kaggle: **Code → + New Notebook → Import from GitHub URL**  \
   URL: `https://github.com/TigreGotico/ww-trainer/blob/dev/notebooks/kaggle_quickstart.ipynb`
3. In the sidebar, click **Session options → Accelerator** and choose **GPU T4 × 1**  \
   (free, no credit card needed; the full run takes ≈ 25–40 min on a T4).
4. Hit **Run All**. That is it for a first run.

**Optional — MLflow tracking:** if you have an MLflow server, store your token as a Kaggle
Secret named `MLFLOW_TOKEN` (Sidebar → Add-ons → Secrets) and set `MLFLOW_URI` in Cell 2.

### Running on Google Colab

Click the **Open in Colab** badge above.  \
Colab and Kaggle are nearly identical for this notebook.  Differences:

- Select the **T4 GPU** runtime (Runtime → Change runtime type).
- Cell 4 detects Colab and mounts Google Drive so outputs survive session resets.
- Kaggle Secrets code in Cell 4 is silently skipped on Colab — no errors.

### Running locally (CPU)

Set `TIER=micro`, `N_POSITIVE=100`, `DOWNLOAD_AUGMENT=false` to cut runtime to ~30–60 min.  \
Install: `pip install "wakeforge[datagen,torchcodec]"`

---

## What this notebook does

| Cell | Step | What happens | Typical time (Kaggle GPU T4) |
|------|------|-------------|-------------------------------|
| 2 | **Configure** | Set your wake phrase and all options | instant |
| 3 | **Install** | Install `ww_trainer` and audio dependencies | 3–5 min |
| 4 | **Platform setup** | Mount Colab Drive; inject Kaggle secrets | 1 min |
| 5 | **Generate dataset** | TTS synthesis + HuggingFace negative download | 15–20 min |
| 6 | **Train** | Train a compact model and export to ONNX | 5–15 min |
| 7 | **Verify ONNX** | Confirm both required files exist | instant |
| 8 | **Inference check** | Confidence-score sanity test | instant |
| 9 | **Ship it** | OVOS config snippet and download guide | instant |

**Total: ≈ 25–40 min on Kaggle GPU T4.** CPU: 60–120 min with reduced settings.

---

## Output files

```
ww_output/
├── dataset/
│   ├── train/metadata.csv         ← training data manifest
│   └── test/metadata.csv          ← evaluation data manifest
└── model/
    ├── best_f1.pt                 ← PyTorch checkpoint (for further fine-tuning)
    ├── best_f1_featurizer.onnx    ← audio feature extractor  ╮ both needed
    └── best_f1.onnx               ← classifier head          ╯ for inference
```

Both ONNX files together are typically under 1 MB for the `small` tier.

---

## Tier reference

| Tier | Architecture | Approx. params | Target device |
|------|-------------|---------------|---------------|
| `micro` | MFCC-40 + FFN | ~50 K | Microcontroller, RPi Zero |
| **`small`** *(default)* | MFCC-40 + GRU | ~200 K | **RPi 3/4, most SBCs** |
| `sincnet_small` | SincNet + GRU | ~300 K | Devices with a bit more headroom |
| `filterbank_small` | FilterBank + GRU | ~250 K | Embedded SBC |
| `gammatone_small` | Gammatone + GRU | ~250 K | Embedded SBC |

If you are unsure, keep `small` — it runs at ~5 % CPU on a Raspberry Pi 4.

---

## Configuration reference

| Variable | Default | Description |
|----------|---------|-------------|
| `WAKE_WORD` | `hey jarvis` | Your phrase — any language, any words |
| `OUTPUT_DIR` | `./ww_output` | Where all outputs land |
| `TIER` | `small` | Model tier (see table above) |
| `EPOCHS` | `50` | Training epochs — more = better, slower |
| `BATCH_SIZE` | `16` | Reduce to `8` if you see out-of-memory errors |
| `N_POSITIVE` | `500` | How many TTS utterances to synthesise |
| `LANG` | `en` | BCP-47 language code (`pt`, `de`, `fr`, …) |
| `ADVERSARIAL` | `true` | Generate phonetically-similar hard negatives |
| `DOWNLOAD_AUGMENT` | `true` | Download background noise / music / reverb from HuggingFace |
| `REUSE_DATASET` | `true` | Skip dataset generation if it already exists |
| `DEVICE` | `auto` | `auto`, `cpu`, or `cuda` |
| `SEED` | `42` | Random seed for reproducibility |
| `MLFLOW_URI` | *(empty)* | MLflow server URL (optional) |
| `MLFLOW_SECRET` | `MLFLOW_TOKEN` | Kaggle Secret name holding the MLflow token |


## Cell 2 — Configuration

**This is the only cell you need to edit.**

Change `WAKE_WORD` to your phrase and click **Run All**.  Everything else has sensible
defaults for a first run on Kaggle GPU T4.

Every variable can also be provided as an environment variable or a Kaggle Secret with the
same name — `os.environ.get(...)` picks those up automatically.

> **Language tip:** `LANG` controls which TTS voice is used for synthesis.  \
> `en` = English, `pt` = Portuguese, `de` = German, `fr` = French, etc.  \
> [Edge-TTS supports 70+ locales.](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support)


In [ ]:
import os

# ── Wake phrase — THE ONE THING YOU NEED TO CHANGE ────────────────────────────
WAKE_WORD        = os.environ.get("WAKE_WORD",        "hey jarvis")

# ── Output location ───────────────────────────────────────────────────────────
OUTPUT_DIR       = os.environ.get("OUTPUT_DIR",       "./ww_output")

# ── Model ─────────────────────────────────────────────────────────────────────
TIER             = os.environ.get("TIER",             "small")
EPOCHS           = int(os.environ.get("EPOCHS",       "50"))
BATCH_SIZE       = int(os.environ.get("BATCH_SIZE",   "16"))
DEVICE           = os.environ.get("DEVICE",           "auto")
SEED             = int(os.environ.get("SEED",         "42"))

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",   "500"))
LANG             = os.environ.get("LANG",             "en")
ADVERSARIAL      = os.environ.get("ADVERSARIAL",      "true").lower() == "true"
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT", "true").lower() == "true"
REUSE_DATASET    = os.environ.get("REUSE_DATASET",    "true").lower() == "true"

# ── MLflow (optional — leave blank to skip) ───────────────────────────────────
MLFLOW_URI       = os.environ.get("MLFLOW_URI",       "")
MLFLOW_SECRET    = os.environ.get("MLFLOW_SECRET",    "MLFLOW_TOKEN")

print(f"Wake word : {WAKE_WORD!r}")
print(f"Tier      : {TIER}  |  Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  Device: {DEVICE}")
print(f"Positives : {N_POSITIVE}  |  Lang: {LANG}  |  Augmentation: {DOWNLOAD_AUGMENT}")


## Cell 3 — Install dependencies

Installs `ww_trainer` from PyPI together with:

- `[datagen]` extra — OVOS TTS plugin (edge-tts) and HuggingFace `datasets` library
  used to download negative speech audio.
- `[torchcodec]` extra — audio codec backend required to decode MP3/AAC files from
  the HuggingFace datasets.

If `ww_trainer` is already installed in this session, pip skips re-installation.
Safe to re-run.

> **Expected runtime:** 3–5 min on Kaggle (packages not pre-cached).


In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Core scientific stack (already on Kaggle/Colab — listed for local runs)
_pip("torch", "torchaudio", "onnx", "onnxruntime",
     "numpy", "soundfile", "librosa", "scikit-learn",
     "matplotlib", "tqdm", "click")

# ww_trainer with data-generation and audio-codec extras
_pip("wakeforge[datagen,torchcodec]")

# Detect running platform
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "colab"      if "google.colab" in sys.modules else
    "paperspace" if os.path.exists("/notebooks")  else
    "local"
)

import ww_trainer
print(f"Platform  : {_platform}")
print(f"ww_trainer: {ww_trainer.__version__}")
print(f"Python    : {sys.version.split()[0]}")


## Cell 4 — Platform setup (Colab Drive + Kaggle Secrets)

This cell handles two platform-specific tasks:

**On Google Colab:** mounts Google Drive and redirects `OUTPUT_DIR` to
`/content/drive/MyDrive/ww_output` so output files survive session resets.
You will be prompted once to authorise Drive access.

**On Kaggle:** reads the MLflow token from Kaggle Secrets (if configured) and injects
it as an environment variable.  Training works fine if no secret exists — this is
silently skipped.

**On any platform:** applies `MLFLOW_URI` if set in Cell 2.

> MLflow is entirely optional.  Skipping it has no effect on the trained model.


In [ ]:
import os

# ── Colab: mount Drive for persistent storage ─────────────────────────────────
if _platform == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/ww_output"
        print(f"Colab: outputs redirected to {OUTPUT_DIR} (survives session resets)")
    except Exception as e:
        print(f"Drive mount skipped ({e}). Outputs will be lost on session reset.")

# ── Kaggle: inject MLflow token from Secrets ──────────────────────────────────
if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"Kaggle: MLflow token injected from Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"Kaggle: no Secret '{MLFLOW_SECRET}' found — MLflow disabled. ({e})")

# ── Any platform: apply MLflow URI if provided ────────────────────────────────
if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    print(f"MLflow URI: {MLFLOW_URI}")
else:
    print("MLflow URI not set — experiment tracking disabled (training still works fine).")


## Cell 5 — Generate the training dataset

This cell builds the labelled audio dataset your model will train on.

**What happens inside:**

1. **Positives** — edge-tts synthesises `N_POSITIVE` utterances of your wake phrase in
   multiple voices, speeds, and pitches.  Each is labelled `1` ("wake word").
2. **Negatives** — short speech clips that do *not* contain the phrase are downloaded
   from public HuggingFace datasets.  Each is labelled `0` ("not wake word").
3. **Adversarial negatives** (when `ADVERSARIAL=true`) — phonetically-similar phrases
   ("hay janice", "hey jarvi") are synthesised as hard negatives, making the model
   more robust to near-misses.
4. **Augmentation audio** (when `DOWNLOAD_AUGMENT=true`) — background noise, music, and
   room impulse responses are downloaded.  They are mixed in at training time to simulate
   real-world listening conditions.

**`REUSE_DATASET=true` (default):** if the dataset already exists at
`OUTPUT_DIR/dataset/`, this cell completes instantly — no re-downloads.
Re-run the whole notebook freely without waiting for downloads every time.

> **Expected runtime:** 15–20 min on Kaggle (mostly HuggingFace downloads).  \
> Set `DOWNLOAD_AUGMENT=false` to cut this to ~5 min (slight accuracy trade-off).

> **Troubleshooting:**
> - *`ModuleNotFoundError: datasets`* — re-run Cell 3 (install did not complete).
> - *`AssertionError: Only X.X GB free`* — free up disk in the Kaggle Output tab.
> - *`vadonnx` fails to load* — run `!pip install -q git+https://github.com/TigreGotico/vadonnx.git`
>   then re-run this cell.


In [ ]:
import shutil
from pathlib import Path
from ww_trainer.datagen import DatagenConfig, DatagenResult, normalize_wake_word, run_datagen_pipeline

# ── Disk space guard ──────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, (
    f"Only {free_gb:.1f} GB free — need at least 3 GB. "
    "Free up space or set DOWNLOAD_AUGMENT=false to reduce requirements."
)
print(f"Disk free: {free_gb:.1f} GB")

# ── Dataset paths ─────────────────────────────────────────────────────────────
dataset_dir        = Path(OUTPUT_DIR) / "dataset"
train_csv_expected = dataset_dir / "train" / "metadata.csv"
test_csv_expected  = dataset_dir / "test"  / "metadata.csv"

if REUSE_DATASET and train_csv_expected.exists() and test_csv_expected.exists():
    # Reconstruct result object from existing paths (no downloads)
    print(f"Reusing existing dataset at {dataset_dir}")
    slug = normalize_wake_word(WAKE_WORD)
    _dr = DatagenResult(
        train_csv=train_csv_expected,
        test_csv=test_csv_expected,
        positives_dir=dataset_dir / slug / "positives",
        negatives_dir=dataset_dir / slug / "negatives",
        bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
        music_dir=dataset_dir / "augmentation" / "music",
        rir_dir=dataset_dir / "augmentation" / "rir",
    )
else:
    print(f"Generating dataset for {WAKE_WORD!r} — this will take 15–20 min...")
    _dr = run_datagen_pipeline(DatagenConfig(
        wake_word=WAKE_WORD,
        output_dir=dataset_dir,
        n_positive=N_POSITIVE,
        lang=LANG,
        adversarial=ADVERSARIAL,
        vad_trim=True,
        download_augmentation=DOWNLOAD_AUGMENT,
        seed=SEED,
    ))

print(f"Train CSV : {_dr.train_csv}")
print(f"Test CSV  : {_dr.test_csv}")
if _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
    print(f"BG noise  : {_dr.bg_noise_dir}")
print("Dataset ready.")


## Cell 6 — Train

Calls `train_from_wakeword()`, which:

1. Detects and reuses the dataset built in Cell 5 (`reuse_dataset=True`).
2. Instantiates the model architecture specified by `TIER`.
3. Trains for `EPOCHS` epochs, saving the best checkpoint by F1 score.
4. Exports two ONNX files: the audio feature extractor and the classifier head.

**Outputs land in `OUTPUT_DIR/model/`.**

The augmentation directories prepared in Cell 5 (background noise, music, reverb)
are picked up automatically — no extra arguments are needed here.

> **Expected runtime:** 5–15 min on Kaggle GPU T4 (50 epochs, `small` tier).

> **Troubleshooting:**
> - *CUDA out of memory* — reduce `BATCH_SIZE` to `8` in Cell 2, re-run from Cell 6.
> - *F1 stays very low (< 0.5)* — increase `N_POSITIVE` to 1000 and re-run from
>   Cell 5 with `REUSE_DATASET=false`.  More data usually fixes this.
> - *Kaggle session timeout (9 h limit)* — the best checkpoint is saved after every
>   epoch.  Re-open the notebook, run Cells 2–5 (dataset reuse is instant), then
>   re-run this cell — training resumes from `best_f1.pt` automatically.


In [ ]:
from ww_trainer.quickstart import train_from_wakeword

print(f"Training: wake_word={WAKE_WORD!r}  tier={TIER!r}  epochs={EPOCHS}  "
      f"batch_size={BATCH_SIZE}  device={DEVICE!r}")

# Note: augmentation dirs from Cell 5 are picked up automatically
# when reuse_dataset=True — no need to pass them explicitly here.
result = train_from_wakeword(
    WAKE_WORD,
    OUTPUT_DIR,
    tier=TIER,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    n_positive=N_POSITIVE,
    lang=LANG,
    adversarial=ADVERSARIAL,
    download_augmentation=DOWNLOAD_AUGMENT,
    reuse_dataset=True,   # dataset was built in Cell 5
)

print("\nTraining complete.")
print(f"  best_onnx_path  : {result.best_onnx_path}")
print(f"  best_model_path : {result.best_model_path}")
print(f"  metrics         : {result.metrics}")


## Cell 7 — ONNX export check

Verifies that both required ONNX files were written successfully.

**Why two files?**

- `best_f1_featurizer.onnx` — converts raw 16 kHz audio to a feature vector
  (MFCC, filterbank, etc.).
- `best_f1.onnx` — takes that feature vector and outputs a confidence score in `[0, 1]`.

Both must be copied to your device together.  At inference time they are chained:

```
raw audio → featurizer.onnx → features → classifier.onnx → score
```

Neither file requires PyTorch at inference time — only `onnxruntime` and `numpy`.

If either file is missing, check the training log above for errors.


In [ ]:
from pathlib import Path

model_dir  = Path(OUTPUT_DIR) / "model"
_feat_onnx = model_dir / "best_f1_featurizer.onnx"
_head_onnx = model_dir / "best_f1.onnx"

print("ONNX files:")
for label, path in [("featurizer", _feat_onnx), ("classifier", _head_onnx)]:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  OK      {label}: {path.name}  ({size_kb:.0f} KB)")
    else:
        print(f"  MISSING {label}: {path}")

assert _feat_onnx.exists() and _head_onnx.exists(), (
    "One or both ONNX files are missing. Check the training output above for errors."
)
print("\nBoth ONNX files present — ready for deployment.")


## Cell 8 — Inference sanity check

Loads both ONNX files with `OnnxWakeWordInferencer` — the same class used at runtime
in OVOS — and scores one positive sample from the test set.

This test uses only `onnxruntime` and `numpy`; no PyTorch is needed.

**Interpreting the score:**

| Score | Meaning |
|-------|---------|
| ≥ 0.8 | Model is confident — ready to deploy |
| 0.5–0.8 | Borderline — consider more epochs or more data |
| < 0.5 | Low confidence — increase `N_POSITIVE` to 1000, `EPOCHS` to 100, re-run |

The default detection threshold in the OVOS plugin is `0.5` (configurable).


In [ ]:
import csv, numpy as np, torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

inferencer = OnnxWakeWordInferencer(str(_feat_onnx), str(_head_onnx))
print("OnnxWakeWordInferencer loaded (onnxruntime only — no PyTorch at runtime).")

# Find a positive sample from the test split
_pos_path = None
with open(result.test_csv) as _f:
    for _row in csv.reader(_f):
        if len(_row) >= 2 and _row[1].strip() == "1" and Path(_row[0]).exists():
            _pos_path = _row[0]
            break

if _pos_path:
    _wav, _sr = torchaudio.load(_pos_path)
    if _sr != 16000:
        _wav = torchaudio.functional.resample(_wav, _sr, 16000)
    _wav_np = _wav.mean(0).numpy().astype(np.float32)
    _score  = inferencer.infer(_wav_np)
    _verdict = (
        "PASS — model confident"                     if _score >= 0.8 else
        "BORDERLINE — consider more data or epochs"  if _score >= 0.5 else
        "LOW — needs more data or training"
    )
    print(f"\nPositive sample : {Path(_pos_path).name}")
    print(f"Confidence score: {_score:.4f}  ({_verdict})")
else:
    print("No positive test sample found in CSV — skipping inference check.")


## Cell 9 — Ship it to your assistant

The two ONNX files are everything needed.  Here is how to get them from this session
onto your device and into OpenVoiceOS.

### Step 1 — Download the files

**On Kaggle:** go to the **Output** tab (right sidebar) → navigate to
`ww_output/model/` → download `best_f1.onnx` and `best_f1_featurizer.onnx`.

**On Colab:** both files are in your Google Drive under
`MyDrive/ww_output/model/` — accessible from any browser.

### Step 2 — Copy to your OVOS device

```bash
scp best_f1_featurizer.onnx best_f1.onnx  \
    ovos@mydevice:~/.local/share/mycroft/precise/
```

### Step 3 — Configure OVOS (ovos-ww-plugin-precise-onnx)

Install the plugin if not already present:

```bash
pip install ovos-ww-plugin-precise-onnx
```

Add the following to `~/.config/mycroft/mycroft.conf` on your device (replace
`hey_jarvis` and the paths with your actual phrase and locations):

```json
{
  "listener": {
    "wake_word": "hey_jarvis"
  },
  "hotwords": {
    "hey_jarvis": {
      "module": "ovos-ww-plugin-precise-onnx",
      "model": "/home/ovos/.local/share/mycroft/precise/best_f1.onnx",
      "sensitivity": 0.5,
      "trigger_level": 3,
      "listen": true
    }
  }
}
```

- `sensitivity` — detection threshold `[0, 1]`.  `0.5` is the default.  \
  Raise toward `0.7` to reduce false activations; lower toward `0.3` for higher recall.
- `trigger_level` — consecutive chunks above threshold required to fire.  `3` avoids
  single-frame false positives.

### Step 4 — Restart and test

```bash
systemctl --user restart ovos-listener
journalctl --user -u ovos-listener -f   # watch for "WW activated"
```

---

### Want a better model?

| Goal | What to change |
|------|----------------|
| Fewer false activations | Raise `EPOCHS` to 100; add real far-field recordings |
| Works well in noisy rooms | Ensure `DOWNLOAD_AUGMENT=true`; raise `N_POSITIVE` to 1000+ |
| Even smaller file (< 100 KB) | Use `TIER=micro` |
| Targeting ESP32 / MCU | See `docs/guides/embedded.md` for the C export path |
| Compare architectures and losses | Use `notebooks/kaggle_experiments.ipynb` |


In [ ]:
import json
from pathlib import Path

_slug      = WAKE_WORD.lower().replace(" ", "_")
_model_dir = Path(OUTPUT_DIR) / "model"
_feat_path = _model_dir / "best_f1_featurizer.onnx"
_head_path = _model_dir / "best_f1.onnx"

print("=" * 60)
print(f"Wake word : {WAKE_WORD!r}")
print(f"Tier      : {TIER}")
print(f"Metrics   : {result.metrics}")
print()
print("Output ONNX files:")
for f in sorted(_model_dir.glob("*.onnx")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
print("=" * 60)

# OVOS config snippet
_ovos_cfg = {
    "listener": {"wake_word": _slug},
    "hotwords": {
        _slug: {
            "module": "ovos-ww-plugin-precise-onnx",
            "model": f"/home/ovos/.local/share/mycroft/precise/best_f1.onnx",
            "sensitivity": 0.5,
            "trigger_level": 3,
            "listen": True
        }
    }
}
print()
print("OVOS mycroft.conf snippet (update paths to your device):")
print(json.dumps(_ovos_cfg, indent=2))
print()
print("Quick Python inference test:")
print("  from ww_trainer.inference import OnnxWakeWordInferencer")
print(f"  model = OnnxWakeWordInferencer({str(_feat_path)!r}, {str(_head_path)!r})")
print("  score = model.infer(wav_float32_16khz_array)  # float in [0, 1]")
